# 03 — Logit-Based Uncertainty

Token probabilities from the LLM provide uncertainty signals without requiring additional model calls.

## Token Probability as Confidence

For autoregressive LLMs, each token is generated with a probability distribution. These distributions can be aggregated into sequence-level uncertainty estimates.

**Sequence probability**: the joint probability of a response is the product of token probabilities:
$$P(y_1, ..., y_T|x) = \prod_{t=1}^T P(y_t | y_{<t}, x)$$
This underestimates confidence for long sequences (product of many probabilities < 1 even for correct tokens). **Length-normalised log probability** is more practical:
$$\hat{P} = \frac{1}{T} \sum_t \log P(y_t | y_{<t}, x)$$

**Per-token uncertainty signals**:
- *Entropy* of the token distribution at each position: $H_t = -\sum_v p_v \log p_v$
- *Max probability* (top-1 prob): simple but correlates well with accuracy
- *Top-1 - Top-2 margin*: gap between most and second-most likely tokens; high gap = confident

**Limitations**: token probabilities reflect the language model's calibration, which may not match factual accuracy. The model can be fluent but wrong, producing high token probabilities for incorrect statements.

In [ ]:
# Sequence probability uncertainty estimator
import numpy as np
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

# Simulate an autoregressive LLM generating sequences
# Returns per-token log probs and whether the answer is correct

class SimulatedLLM:
    def __init__(self, vocab_size=50, base_accuracy=0.7):
        self.vocab_size = vocab_size
        self.base_accuracy = base_accuracy

    def generate(self, question_idx, seq_len=5):
        """Returns token probs and whether answer is correct."""
        np.random.seed(question_idx)
        # Questions the model knows well vs poorly
        difficulty = np.random.uniform(0.3, 1.0)
        is_correct = np.random.random() < difficulty

        # Generate per-token probabilities
        token_probs = []
        for _ in range(seq_len):
            # High difficulty -> more peaked distribution
            alpha = difficulty * 5
            # Most mass on correct token
            probs = np.random.dirichlet([alpha] + [0.3] * (self.vocab_size - 1))
            probs = np.sort(probs)[::-1]  # sorted descending
            token_probs.append(probs)

        return token_probs, is_correct, difficulty

llm = SimulatedLLM()

# Compute uncertainty metrics for many questions
results = []
for q_idx in range(200):
    token_probs, is_correct, difficulty = llm.generate(q_idx)

    # Metrics
    log_probs = np.array([np.log(tp[0] + 1e-10) for tp in token_probs])
    mean_log_prob = log_probs.mean()

    entropies = np.array([-np.sum(tp * np.log(tp + 1e-10)) for tp in token_probs])
    mean_entropy = entropies.mean()

    margin = np.array([tp[0] - tp[1] for tp in token_probs]).mean()

    results.append({
        'mean_log_prob': mean_log_prob,
        'mean_entropy': mean_entropy,
        'margin': margin,
        'is_correct': int(is_correct),
        'difficulty': difficulty,
    })

# Correlation analysis
import numpy as np
from scipy.stats import pearsonr

for metric in ['mean_log_prob', 'mean_entropy', 'margin']:
    vals = [r[metric] for r in results]
    corrects = [r['is_correct'] for r in results]
    corr, p = pearsonr(vals, corrects)
    print(f'{metric:<20}: r={corr:.3f} (p={p:.4f})')

In [ ]:
# Calibration of sequence probability scores
import matplotlib.pyplot as plt

mean_log_probs = np.array([r['mean_log_prob'] for r in results])
corrects = np.array([r['is_correct'] for r in results])

# Convert log probs to confidence scores via sigmoid
# Normalise to [0,1]
lp_norm = (mean_log_probs - mean_log_probs.min()) / (mean_log_probs.max() - mean_log_probs.min())

# Bin by confidence and compute accuracy
n_bins = 10
edges = np.linspace(0, 1, n_bins + 1)
bin_accs, bin_confs = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    mask = (lp_norm >= lo) & (lp_norm < hi)
    if mask.sum() > 0:
        bin_accs.append(corrects[mask].mean())
        bin_confs.append(lp_norm[mask].mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax1.scatter(bin_confs, bin_accs, color='steelblue', s=60)
ax1.plot(bin_confs, bin_accs, color='steelblue', alpha=0.5)
ax1.set_xlabel('Normalised sequence probability')
ax1.set_ylabel('Accuracy')
ax1.set_title('Logit-Based Reliability Diagram')
ax1.legend()

ax2.hist([mean_log_probs[corrects == 1], mean_log_probs[corrects == 0]],
          bins=20, label=['Correct', 'Wrong'], alpha=0.7, color=['steelblue', 'tomato'])
ax2.set_xlabel('Mean log probability')
ax2.set_ylabel('Count')
ax2.set_title('Log Prob Distribution: Correct vs Wrong')
ax2.legend()

plt.tight_layout()
plt.savefig('/tmp/logit_uncertainty.png', dpi=100, bbox_inches='tight')
plt.show()
print('Logit-based uncertainty analysis saved')